# Week 5 へようこそ - エージェントフレームワーク

## Day 1: Google ADK (と A2A のちょっとした紹介)

今週は、今日の主要なエージェントフレームワークを一気に見て回ります。大事な考え方は、ひとつのフレームワークをしっかり理解すれば、他もだいたい理解できるということです。どのフレームワークでも、エージェントを作る手順は同じ5つのステップなので、すぐに見慣れたパターンに感じられるはずです。

1. **エージェントを作る** - モデルとシステムプロンプトを与える。
2. **実行する** - メッセージを送り、返信を受け取る。
3. **ツールを追加する** - エージェントが呼び出せる、普通の型付き関数。
4. **MCP を追加する** - 誰か他の人が書いたツールサーバーに接続する。毎回同じ方法でつなげる。
5. **ゴールを与えてループさせる** - 目標を渡し、仕事が終わるまで一歩ずつ自分で進めさせる。

ステップ1と2は、まだ単なる LLM 呼び出しです。ツールと MCP は、エージェントにできることを与えます。ステップ5でようやくエージェントらしくなります。フレームワーク自身がループを回し、ツールを選び、結果を読み、また選び直す、というのをゴールに到達するまで続けるのです。このループこそ、今週ずっと注目すべきものです。

まずは Google ADK から始めます。今週いちばん時間をかけるフレームワークです。開発者体験がもっとも洗練されていて、リアルタイムで見られるビジュアルトレースがあり、MCP と A2A の両方を第一級のサポートとして備えています。

今週を通しての実習プロジェクトは、小さな SQLite の todo ボードです。ワーカーがボードから1つのタスクを取り出し、自分のエージェントループでそれをこなし、結果を書き込んで、タスクを完了にします。ここでワーカー全体を自分で組み立てて実行し、その後 ADK のビジュアル UI でも同じものを見ます。Day 5 では、この同じボードが成長し、エージェントのチーム全体を統率するようになります。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Google ADK のドキュメント</h2>
            <span style="color:#00bfff;">ドキュメントは <a href="https://adk.dev">https://adk.dev</a> にあり、一度目を通しておく価値があります。ADK には 1.x と 2.x の2つのリリースラインがありますが、ここでは最新の 2.x を使います。古いブログ記事には注意してください。最近いくつかのインポートパスが変更されています。</span>
        </td>
    </tr>
</table>

## セットアップ

今日必要なものは2つですが、どちらも以前の週からすでに用意されています。

- **Node**。`npx` のために必要です(filesystem MCP サーバーはこれを使って動きます)。`node --version` で確認してください。
- リポジトリのルートにある `.env` の中の **`GOOGLE_API_KEY`**。これは CrewAI の週で設定済みなので、もう存在しているはずです。

Google ADK はリポジトリの環境に含まれているので、リポジトリのルートで通常の `uv sync` を実行すればすべてインストールされます。このノートブックを Cursor で開き、毎週使っているリポジトリ既定の **Python 3.12.12** カーネルを選んで、上から順にセルを実行してください。

このノートブックはステップ4で、`npx` 経由の filesystem MCP サーバーを自動的に起動します。最初の実行を速くするために、今のうちに一度ウォームアップしておき、動作中と表示されたらすぐに Ctrl-C で止めてください。

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [ ]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from quiet import silence
silence()  # うるさいライブラリのログを抑えて、エージェントのトレースを読みやすくする

os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")
load_dotenv(override=True)

## ステップ1: エージェントを作る

ADK では、エージェントは `LlmAgent` です。モデルと、システムプロンプトである `instruction` を持つだけのオブジェクトで、ラッパークラスも登録手順もありません。まずセットアップから。`GOOGLE_API_KEY` はリポジトリルートの `.env` から読み込まれます。

In [ ]:
MODEL = "gemini-flash-latest"

agent = LlmAgent(
    model=MODEL,
    name="assistant",
    instruction="You are a concise, friendly assistant. Reply in a single short sentence.",
)

## ステップ2: 実行する

`InMemoryRunner(...).run_debug(..., verbose=True)` はエージェントを実行し、各ステップを表示してくれるので、何をしているか観察できます。まだツールがないので、ループするものは何もなく、エージェントはただ返信するだけです。これはまだ単なる LLM 呼び出しです。

In [ ]:
result = await InMemoryRunner(agent=agent).run_debug("Say hello in Spanish.", verbose=True)

## 今週のプロジェクト: SQLite の todo ボード

ワーカーは小さな SQLite ボードを介して連携します。1つのファイル、1つのテーブルで、サーバーを立てる必要もありません。`board.py` はいくつかの小さな関数の集まりです。ワーカーには1つの**ゴール**が与えられ、それを達成するために自分自身の**ステップ**の todo をそのゴールの下に書き出し、進めるごとにチェックを入れていき、最後にゴールを完了にします。内部的にはボードは単なる辞書のリストです(ゴールの `parent_id` は None で、ステップは自分のゴールを指します)。

In [ ]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

`show_board()` は、Week 1 で使ったのと同じ rich スタイルで、その同じデータを綺麗に表示します。各ゴールの下にステップがインデントされて並び、完了したタスクは緑色の打ち消し線、進行中のタスクは黄色で表示されます。まだステップはありません。エージェントが計画を立てるときに自分でステップを書き出します。

In [ ]:
board.show_board()

## ステップ3: ツールを追加する

ADK でのツールは、docstring 付きの型付き Python 関数にすぎません。docstring はモデルが読む説明文になり、型ヒントは引数のスキーマになります。そして、その関数をエージェントの `tools=[...]` リストに渡すだけです。

ここでは3つの小さなボードツールを書きます。ボードを読む `show_todos`、ゴールをステップに分解する `plan_steps`、todo を完了にする `complete_task` です。まずは簡単なエージェントに2つだけ与えて、ボードに何があるか尋ねてみましょう。答える前に自分から `show_todos` を呼び出すことを、自分の目で確かめてください。この「決める、呼ぶ、読む、答える」というサイクルこそ、エージェントループが回り始めた瞬間です。3つのツールすべてはステップ5で一緒になります。

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}


In [ ]:
board_agent = LlmAgent(
    model=MODEL,
    name="board_agent",
    instruction="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [ ]:
result = await InMemoryRunner(agent=board_agent).run_debug("What is on the board right now, and what is its status?", verbose=True)

### ツールは実世界にも届く

ボードツールは小さなデータベースとやり取りするだけでしたが、ツールは同じくらい簡単にマシンの外の世界にも届きます。ここでは、Pushover を通じてスマートフォンにプッシュ通知を送るツールを紹介します。これはコースの他の場所でも使われている `send_push_notification` そのもので、やはり docstring 付きの型付き関数にすぎません。ADK はこれを読んで呼び出し方を学びます。

In [ ]:
import requests

def send_push_notification(message: str) -> str:
    """Send a short push notification to the user's phone to report a result or that a job is done."""
    payload = {
        "token": os.getenv("PUSHOVER_TOKEN"),
        "user": os.getenv("PUSHOVER_USER"),
        "message": message,
        "sound": "climb"
    }
    response = requests.post("https://api.pushover.net/1/messages.json", data=payload)
    return f"Push notification sent (status {response.status_code})."

In [ ]:
notifier = LlmAgent(
    model=MODEL,
    name="notifier",
    instruction="You notify the user. When asked, use your tool to send a push notification.",
    tools=[send_push_notification],
)

result = await InMemoryRunner(agent=notifier).run_debug("Send a push notification that says hello from Google ADK.", verbose=True)

## ステップ4: MCP を追加する

MCP は、単に「自分が書いていないツール」を、小さなプロトコル越しに接続したものです。今週すべてのフレームワークで使う同じ Node サーバーである filesystem リファレンスサーバーを、単一の `workspace` フォルダに限定してエージェントに与えます。これにより、エージェントはそのフォルダ内のファイルしか触れなくなります。ADK では、これは同じ `tools=[...]` リストに追加する1つの `McpToolset` にすぎません。

ADK はこのサーバーを、`npx` 経由の小さな Node プロセスとして自動的に起動します。知っておく価値のある詳細が1つあります。`errlog=subprocess.DEVNULL` を渡している点です。これによりサーバーの起動時ログが静かになるだけでなく、Windows 上の Jupyter カーネルからサーバーを実行できるようにもなります。Windows のカーネルの stderr には、サーバーが書き込める実際のファイルディスクリプタがないためです。Mac と Linux では、これは何も変わりません。

In [ ]:
import subprocess
from pathlib import Path
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

workspace = Path("task_worker/workspace").resolve()   # エージェントが触れてよい唯一のフォルダ

filesystem = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),  # 相対ファイル名がそこで解決されるよう、workspace内でサーバーを起動する
        ),
        timeout=60
    ),
    errlog=subprocess.DEVNULL,
)

In [ ]:
file_agent = LlmAgent(
    model=MODEL,
    name="file_agent",
    instruction="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem],
)

In [ ]:
result = await InMemoryRunner(agent=file_agent).run_debug("Read notes.txt and summarize it in one short sentence.", verbose=True)

## ステップ5: ゴールを与えてループさせる

さあ、いよいよ本番です。1つのエージェントに3つのボードツールすべてと filesystem サーバーを与え、ゴールを渡して、実行させましょう。エージェントは自分でボード上にステップを計画し、ファイルツールでそれを片付け、それぞれにチェックを入れ、作業が終わったらゴールを完了にします。これこそ、自律的に動くエージェントループです。読む、計画する、行動する、チェックする、繰り返す。

今回は `run_debug` ではなく `run_async` でループを駆動し、イベントストリームを自分たちで処理します。エージェントがツールを呼ぶたびにそれを表示し、ステップが計画されたりチェックされたりするたびにボードを再描画するので、計画が現れてから一行ずつ打ち消し線が引かれていく様子を観察できます。

In [ ]:
from google.genai import types

INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = LlmAgent(
    model=MODEL,
    name="task_worker",
    instruction=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem, send_push_notification],
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt and send a push notification with a summary.")
board.claim_todo(goal_id)

# run_debug の代わりに run_async でループを自分たちで駆動し、観察できるようにする:
# 各ツール呼び出しを表示し、ステップが計画されたりチェックされたりするたびにボードを再描画する。
runner = InMemoryRunner(agent=worker)
session = await runner.session_service.create_session(app_name=runner.app_name, user_id="ed")

async for event in runner.run_async(
    user_id="ed",
    session_id=session.id,
    new_message=types.UserContent("Please work the pending goal on the board."),
):
    for call in event.get_function_calls():
        print(f"tool: {call.name}({call.args})")
    if any(r.name in ("plan_steps", "complete_task") for r in event.get_function_responses()):
        board.show_board()
    if event.is_final_response():
        print(event.content.parts[0].text)

## どの Runner を、どの run メソッドで?

少し立ち止まって理解しておく価値があります。どの ADK プロジェクトでも、この両方に出会うことになるからです。

**Runner について。** ここでは `InMemoryRunner` を使いました。これはセッション、アーティファクト、メモリの各サービスがすべてメモリ上にある `Runner` です。何も保存されず、カーネルを止めればすべてリセットされるので、学習や実験には理想的です。実際のアプリで使うのは、素の `Runner` です。同じエージェントに、データベースを裏に持つセッションサービスなどの本物のサービスを組み合わせることで、会話が実行間・ユーザー間で永続化されます。

**run メソッドについて。** ステップ2から4では `run_debug` を使いました。これはセッションを作り、メッセージを送り、トレースを表示し、イベントのリストを返す便利なメソッドです。ADK自身のドキュメントも、これはデバッグと実験専用だとしています。ステップ5では、本来のエントリーポイントである `run_async` に切り替えました。ここではセッションを自分で管理し、イベントストリームを自分で処理します。これによって、各ツール呼び出しを表示し、ループが動く様子に合わせてボードを再描画することができたのです。

```python
from google.genai import types

runner = InMemoryRunner(agent=agent)
session = await runner.session_service.create_session(app_name=runner.app_name, user_id="ed")
message = types.UserContent("Say hi in French.")

async for event in runner.run_async(user_id="ed", session_id=session.id, new_message=message):
    print(event.author, event.content)
```

`run_debug` はこれらすべてを隠してエージェントに集中できるようにし、`run_async` はステップ5のように、制御を自分の手に取り戻したいときにそれを返してくれます。他にも、非同期でないコード向けに `run_async` を素朴な同期版にした `run`、そしてリアルタイムの音声・映像向けの実験的なストリーミングモードである `run_live` があります。

## 同じワーカーを、ADK プロジェクトとして

ここまで、ワーカー全体をこのノートブックの中で組み立ててきました。`task_worker/` フォルダは、まさにそれをモジュールとしてパッケージ化したものです。`__init__.py` と `agent.py` に、同じボードツール、同じ filesystem サーバー、同じ instruction が収められています。フォルダとしてパッケージ化することで、ADK のツール群がそれを読み込めるようになります。ADK のエージェントとは、まさにこのようなフォルダのことで、`adk create <name>` でゼロから新しいものをスキャフォールドすることもできます。

モジュール化すると、無料で2つのことが手に入ります。まず、素のスクリプトとして実行できます。これはゴールを1つ登録して、上と同じようにループを実行します。この日のフォルダでターミナルを開いて実行してください。

```bash
uv run worker.py
```

2つ目、そしてこれこそ ADK が今週いちばん洗練された開発者体験である理由ですが、ライブのビジュアルトレースで観察できます。

```bash
uv run worker.py --seed-only   # 新しいゴールをボードに登録する
uv run adk web                 # 表示されたリンクを開く
```

ドロップダウンから **task_worker** を選んでください(`a2a_demo` も表示されますが、これは下のA2Aのセクション用なので、ここでは無視してください)。そして「please work the pending goal on the board」と入力します。ループが動くたびに、すべてのツール呼び出しと MCP 呼び出しがタイムライン上で点灯します。ここで観察したのと同じループが、今度は UI 付きで見られます。

## A2A のちょっとした紹介

A2A (Agent2Agent) は、別々のエージェントが HTTP 越しに互いを発見し呼び出せるようにするための標準です。ADK はこれを2つの呼び出しでサポートします。`to_a2a` はエージェントをサービスとして公開し、`RemoteA2aAgent` はリモートのエージェントをローカルなサブエージェントのように呼び出します。`a2a_demo/` フォルダには小さな例があります。サービスとして公開された翻訳エージェントと、翻訳をそれに委任するローカルなコンシェルジュです。

まず、前のセクションの `adk web` がまだ動いているなら、Ctrl-C で止めてください。このデモは `a2a_demo` の中から独自の `adk web` を起動しますが、`adk web` はそれを起動したフォルダの中のエージェントしか一覧しません。そこから起動することで `spanish_concierge` がドロップダウンに現れます。日のルートから起動すると、代わりに `task_worker` が見えるだけです。

`a2a_demo` フォルダの中で、2つのターミナルを使います。

```bash
cd a2a_demo

# ターミナル1: 翻訳エージェントを A2A サービスとしてポート8001で公開する
uv run uvicorn server:a2a_app --host localhost --port 8001

# ターミナル2: 翻訳エージェントの agent card を取得し、コンシェルジュを操作する
curl http://localhost:8001/.well-known/agent-card.json
uv run adk web   # spanish_concierge を選び、何かをスペイン語に翻訳するよう頼んでみる
```

コンシェルジュはどうやって翻訳エージェントを見つけるのでしょうか。実は、どこを見ればいいかをこちらから教えているのです。`spanish_concierge/agent.py` では、`RemoteA2aAgent` が翻訳エージェントの card の URL、`http://localhost:8001/.well-known/agent-card.json` を使って構築されています。ブロードキャストや自動スキャンは一切なく、A2A クライアントは単に既知の card の URL を指し示されるだけです。上の `curl` は、まさにその card を取得しています。これによってコンシェルジュは、1つのリクエストを送る前に、そのエージェントの名前、できること、そして到達方法を学ぶのです。その後コンシェルジュは A2A 経由で翻訳を委任し、スペイン語の結果が別のプロセスから返ってきます。

A2A は本物の Linux Foundation の標準であり、その発見の仕組みはよく整理されていますが、実際の採用状況は世間の期待にかなり遅れていて、日々のエージェント開発を担っているのは今のところ MCP です。A2A は Day 5 のプロジェクトには再登場しません。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">エクササイズ</h2>
            <span style="color:#ff7800;">ボードに別のゴールを、たとえば「マドリードについての短い俳句を書いて madrid.txt に保存する」を登録し、ワーカーを再度実行してみましょう。ワーカーは適切なステップを計画し、正しいファイルツールを選べるでしょうか。次に、自分自身の4つ目のツール、つまり普通の型付き関数をワーカーに追加し、それがトレースに現れる様子を観察してみましょう。</span>
        </td>
    </tr>
</table>